## tests

In [1]:
from pypdf import PdfReader
reader = PdfReader("/Users/changhungchun/note/1.Research/1.0Literature/行為科學Behavioral Science/傳播學 Communication/CAT/gregory1996nonverbal.pdf")
# num_pages = len(reader.pages)
# page = reader.pages[num_pages - 1]  # Get the last page
text = ""
for page in reader.pages:
    text += page.extract_text()
# print(num_pages)
print(text[:5000])  # Print the first 1000 characters of the text
print("Total characters:", len(text))

Journal of Personality and Social Psychology Copyright 1996 by the American Psychological Association, Inc. 
1996, Vol. 70, No. 6, 1231-1240 0022-3514/96/$3.00 
A Nonverbal Signal in Voices of Interview Partners Effectively Predicts 
Communication Accommodation and Social Status Perceptions 
Stanford W. Gregory, Jr. and Stephen Webster 
Kent State University 
Derivations from nonverbal communications accommodation theory are tested, and this knowledge 
is extended both theoretically and methodologically. Fast fourier transform and statistical analysis of 
a low-frequency nonverbal signal in voices from 25 dyadic interviews between a talk show host and 
his guests revealed voice convergence between partners. Correlation coefficients from comparisons 
of partners' voice spectra and factor analysis of the correlation matrix showed that lower status 
partners accommodated their voices to higher status partners via the nonverbal signal. Student rat- 
ings of the social status of the same ta

In [106]:
import re

def clean_doi(doi):
    """移除DOI結尾的標點符號"""
    return re.sub(r'[^\w/-]+$', '', doi)

# 測試
text = "10.1609/aaai.v31i1.11164."
pattern = r'10\.\d{4,}(?:\.\d+)*/[-._;()\/:a-zA-Z0-9]+(?![a-zA-Z])'
doi = re.search(pattern, text).group()
clean_doi_result = clean_doi(doi)
print(clean_doi_result)  # 10.1145/3107990.3108004

10.1609/aaai.v31i1.11164


## 主程式：

目前，是先從 pdf 中找到 doi，然後把用 doi 請求 bibtex
下一步：
然而，pdf 很可能找不到 doi (20/35)，因此應該要用 llm 找出文章標題，然後用標題找出 doi

In [ ]:
def extract_text_from_pdf(pdf_path):
    """從PDF文件中提取文本"""
    from pypdf import PdfReader
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    print(f"Extracted text from {pdf_path}: {text[:100]}...")  # 打印前100個字符以確認提取成功
    
    return text

def extract_doi(text):
    """從文本中提取DOI"""
    import re
    # DOI的正則表達式模式
    doi_pattern = r'10\.\d{4,}(?:\.\d+)*/[-._;()\/:a-zA-Z0-9]+(?![a-zA-Z])'
    match = re.search(doi_pattern, text)
    if match:
        doi = match.group(0)
        doi = re.sub(r'[^\w/-]+$', '', doi)  # 清理DOI結尾的標點符號
        print(f"Extracted DOI: {doi}")  # 打印提取的DOI
    else:
        doi = None
        print("No DOI found in the text.")
    return doi if doi else None

def get_bibtex_from_doi(doi):
    import requests
    """使用DOI獲取BibTeX"""
    url = f"http://dx.doi.org/{doi}"
    headers = {"Accept": "application/x-bibtex"}
    response = requests.get(url, headers=headers)
    if "<title>Error: DOI Not Found</title>" in response.text:
        print(f"❌ DOI {doi} not found.")
        return "failed"
    else:
        print(f"✅ Retrieved BibTeX for DOI {doi}.")
    return response.text

In [14]:
import openai
import os
import dotenv

# Load environment variables from .env file
dotenv.load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

client = openai.Client()

def get_article_title(file_path):
    """Extracts the title of an article from a PDF file using OpenAI's GPT model.
    Args:
        file_path (str): The path to the PDF file.
    Returns:
        str: The extracted title of the article.
    """
    reader = PdfReader(file_path)
    page = reader.pages[0]
    text = page.extract_text()[:500]
    if not text:
        return "Failed to extract text from the PDF."

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant that extracts text from PDF files, only return pure text of the title.",
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": f"Please extract the title of the article from this {text}"},
                ]
            }
        ],
    )
    return response.choices[0].message.content

import requests
from typing import Optional

crossref_api = "https://api.crossref.org/works"
semantic_scholar_api = "https://api.semanticscholar.org/graph/v1/paper/search"

import time
def search_doi_by_title(title: str, author: str = "") -> Optional[str]:
    """改進的DOI搜尋，遵循Crossref最佳實踐"""
    crossref_api = "https://api.crossref.org/works"
    
    try:
        params = {
            'query.title': title,
            'rows': 1
        }
        if author:
            params['query.author'] = author
        
        # 遵循Crossref禮貌政策
        headers = {
            'User-Agent': 'MyApp/1.0 (mailto:jonathanchang9@gmail.com)',
            'Accept': 'application/json'
        }
        
        # 增加超時，添加email獲得優先處理
        response = requests.get(
            crossref_api, 
            params=params, 
            headers=headers,
            timeout=30
        )
        
        if response.status_code == 429:  # 速率限制
            print("觸發速率限制，等待後重試")
            time.sleep(5)
            return None
        
        data = response.json()
        
        if data['message']['items']:
            return data['message']['items'][0].get('DOI')
            
    except requests.exceptions.Timeout:
        print("Crossref超時，嘗試備用API")
        # return search_doi_openalex(title, author)
    
    except Exception as e:
        print(f"搜尋失敗: {e}")
    
    return None

### 更改 bibtex key

In [ ]:
import re

def parse_bibtex_info(bibtex: str) -> dict:
    """從BibTeX字符串解析資訊"""
    info = {}
    
    # 提取標題
    title_match = re.search(r'title\s*=\s*\{([^}]+)\}', bibtex, re.IGNORECASE)
    info['title'] = title_match.group(1) if title_match else ''

    # 提取作者
    author_match = re.search(r'author\s*=\s*\{([^}]+)\}', bibtex, re.IGNORECASE)
    if author_match:
        authors_str = author_match.group(1)
        # 分割多個作者，處理 "and" 連接
        authors = [a.strip() for a in re.split(r'\s+and\s+', authors_str)]
        info['authors'] = authors
    else:
        info['authors'] = []
    
    # 提取年份
    year_match = re.search(r'year\s*=\s*\{?(\d{4})\}?', bibtex, re.IGNORECASE)
    info['year'] = year_match.group(1) if year_match else ''
    
    return info

def generate_bibtex_key(title: str, authors: list, year: str) -> str:
    """生成kirilyuk2006complex格式的BibTeX key"""
    # 取第一作者姓氏
    first_author = 'unknown'
    if authors:
        # 處理 "LastName, FirstName" 格式
        first_author_name = authors[0].split(',')[0].strip() if ',' in authors[0] else authors[0].split()[-1]
        first_author = re.sub(r'[^a-zA-Z]', '', first_author_name).lower()
    
    # 取標題第一個有意義的詞
    title_words = re.findall(r'\w+', title.lower())
    meaningful_words = [w for w in title_words if w not in ['a', 'an', 'the', 'of', 'in', 'on', 'for', 'with', 'to']]
    title_word = meaningful_words[0] if meaningful_words else 'unknown'
    
    # 清理年份
    year = re.sub(r'[^0-9]', '', str(year))
    
    return f"{first_author}{year}{title_word}"

def customize_bibtex_key(bibtex: str, paper_info=None) -> str:
    """替換BibTeX中的key為自定義格式"""
    if not bibtex:
        return bibtex, "BibTeX string is empty", "No key found in BibTeX"
    
    # 如果沒有提供paper_info，從bibtex中解析
    if paper_info is None:  # 修正：明確檢查None
        parsed_info = parse_bibtex_info(bibtex)
        title = parsed_info['title']
        authors = parsed_info['authors']
        year = parsed_info['year']
    else:
        title = paper_info.title
        authors = paper_info.authors
        year = paper_info.year
    
    # 提取原key
    original_key_match = re.search(r'@\w+\{([^,]+),', bibtex)
    if not original_key_match:
        return bibtex, "No key found in BibTeX", "No key found in BibTeX"
    
    # 生成新key
    new_key = generate_bibtex_key(title, authors, year)
    
    # 替換key
    new_bibtex = bibtex.replace(original_key_match.group(1), new_key)
    return new_bibtex, new_key, title

In [69]:
bibtex = "@article{beike2016is, title={Is sharing specific autobiographical memories a distinct form of self-disclosure?}, volume={145}, ISSN={0096-3445}, url={http://dx.doi.org/10.1037/xge0000143}, DOI={10.1037/xge0000143}, number={4}, journal={Journal of Experimental Psychology: General}, publisher={American Psychological Association (APA)}, author={Beike, Denise R. and Brandon, Nicole R. and Cole, Holly E.}, year={2016}, pages={434–450} }"

line = bibtex.split('},')
print(line[0].split(',')[0])

@article{beike2016is


In [ ]:
def format_bibtex(bibtex: str) -> str:
    """格式化BibTeX使其排版整齊"""
    lines = bibtex.split('},')
    formatted_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # 如果是第一行
        if line.startswith('@'):
            formatted_lines.append(f"{line.split(',')[0]},")
            line = line.split(',')[1]
        # 如果是其他行
        formatted_line = format_content(line)
        # 如果是最後一行    
        if not line.endswith('}'):
            formatted_line = formatted_line + '},'
        formatted_lines.append(formatted_line)    

    return '\n'.join(formatted_lines)
def format_content(line: str) -> str:
    """格式化BibTeX內容"""
    parts = line.split('=', 1)
    field = parts[0].strip()
    value = parts[1].strip().rstrip(',')
    return f"  {field:<12} = {value}"

In [84]:
bibtex = "@article{beike2016is, title={Is sharing specific autobiographical memories a distinct form of self-disclosure?}, volume={145}, ISSN={0096-3445}, url={http://dx.doi.org/10.1037/xge0000143}, DOI={10.1037/xge0000143}, number={4}, journal={Journal of Experimental Psychology: General}, publisher={American Psychological Association (APA)}, author={Beike, Denise R. and Brandon, Nicole R. and Cole, Holly E.}, year={2016}, pages={434–450} }"
formatted_bibtex = format_bibtex(bibtex)
print(formatted_bibtex)

@article{beike2016is,
    title        = {Is sharing specific autobiographical memories a distinct form of self-disclosure?},
    volume       = {145},
    ISSN         = {0096-3445},
    url          = {http://dx.doi.org/10.1037/xge0000143},
    DOI          = {10.1037/xge0000143},
    number       = {4},
    journal      = {Journal of Experimental Psychology: General},
    publisher    = {American Psychological Association (APA)},
    author       = {Beike, Denise R. and Brandon, Nicole R. and Cole, Holly E.},
    year         = {2016},
    pages        = {434–450} }


In [ ]:
def form_template(bibtex: str = "", title: str = "", new_key: str = "") -> str:
  import os
  from datetime import datetime
  CURRENT_YEAR = datetime.now().year
  CURRENT_MONTH = datetime.now().month
  CURRENT_DATE = datetime.now().day
  tags = ""
  if not bibtex:
      bibtex = "No BibTeX provided"
  template = f"""---
start date: {CURRENT_YEAR}/{CURRENT_MONTH}/{CURRENT_DATE}
end date: //
tags:
{tags}
pdf: {new_key}.pdf 
---
# {title}

```bibtex
{bibtex}
```

## 研究背景、動機與目的

### 研究背景

### 研究動機

### 研究目的

## 文獻回顧

## 研究問題

## 研究方法

### 材料

### 實驗設計

### 量測

## 研究結果

## 討論
"""
  return template

In [85]:
import os
def main():
    """主函數，遍歷當前目錄下的所有PDF文件並處理"""
    current_dir = "."
    failed_list_path = "failed_list.md"
    failed_list = "Failed files:\n"
    total = 0
    failed = 0
    for _, dirs, files in os.walk(current_dir):
        for file in files:
            if file.endswith(".pdf"):
                # print(file)
                pdf_path = os.path.join(current_dir, file)
                print(f"Processing {pdf_path}")

                # Extract text from PDF
                content = extract_text_from_pdf(pdf_path)
                doi = extract_doi(content)
                
                if doi is None:
                    print("Using magic.")
                    title = get_article_title(pdf_path)
                    if "Failed" in title:
                        print(f"Failed to extract title from {file}")
                        failed += 1
                        continue
                    print(f"Extracted title: {title}")
                    doi = search_doi_by_title(title)
                    print(f"Found DOI: {doi}")
                
                bibtex = get_bibtex_from_doi(doi)
                
                if bibtex != "failed":
                    print(f"BibTeX for {file}")
                else:
                    failed += 1
                    failed_list += file
                    print(f"❌ Failed to process {file}")
                    continue
                total += 1
                
                print(bibtex)
                # Customize BibTeX key, 寫入 reference.bib
                bibtex, new_key, title = customize_bibtex_key(bibtex)
                formatted_bibtex = format_bibtex(bibtex)
                with open("reference.bib", "a") as f:
                    f.write(formatted_bibtex)
                template = form_template(formatted_bibtex, title)
                with open(f"{new_key}.md", "w") as f:
                    f.write(template)
    
    with open(failed_list_path, "a") as f:
        f.write(failed_list)

    # Summary of results    
    print(f"Total files processed: {total}")
    print(f"Total files failed: {failed}")
    print(f"Success rate: {((total - failed) / total) * 100:.2f}%")

In [87]:
main()

Processing ./mcQuillin2022learning.pdf
Extracted text from ./mcQuillin2022learning.pdf: Learning Socially Appropriate Robo-waiter
Behaviours through Real-time User Feedback
Emily McQuillin...
No DOI found in the text.
Using magic.
Extracted title: Learning Socially Appropriate Robo-waiter Behaviours through Real-time User Feedback
Found DOI: 10.1109/hri53351.2022.9889395
✅ Retrieved BibTeX for DOI 10.1109/hri53351.2022.9889395.
BibTeX for mcQuillin2022learning.pdf
 @inproceedings{McQuillin_2022, title={Learning Socially Appropriate Robo-waiter Behaviours through Real-time User Feedback}, url={http://dx.doi.org/10.1109/hri53351.2022.9889395}, DOI={10.1109/hri53351.2022.9889395}, booktitle={2022 17th ACM/IEEE International Conference on Human-Robot Interaction (HRI)}, publisher={IEEE}, author={McQuillin, Emily and Churamani, Nikhil and Gunes, Hatice}, year={2022}, month=mar, pages={541–550} }

Processing ./kirilyuk2006complex.pdf
Extracted text from ./kirilyuk2006complex.pdf: arXiv:physi